# 10. Futility-анализ


## План занятия

0. **Постановка задачи** — efficacy vs futility
1. **Математическая разработка подхода**
    1. **$Z$ и $B$ значения**
    2. **Conditional Power (CP)**
    3. **Predictive Power (PPoS)**
    4. **β-spending**
        1. Предварительная фиксация futility-границы.
        2. **Binding** и **non-binding** фиксация
2. **Влияние на ошибки I/II рода**
3. **Практические рекомендации.**
4. **Сквозной пример: сравнение всех методов** — $z$-тест, GST, GST + futility (binding / non-binding / CP / PPoS).


## Библиотеки

In [1]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown

pio.templates.default = "plotly_white"

SEED = 12345
rng = np.random.default_rng(SEED)

# Единая палитра исходов на всю лекцию
C_FUT = "#d95f02"   # оранжевый — futility
C_EFF = "#1b9e77"   # зелёный — efficacy
C_CONT = "#999999"  # серый — продолжаем


def style(fig, height=440, title=None, xlabel=None, ylabel=None):
    """Единый стиль всех графиков: сетка, легенда, подписи."""
    fig.update_layout(
        template="plotly_white",
        height=height,
        title=dict(text=title, font=dict(size=15)),
        xaxis_title=xlabel, yaxis_title=ylabel,
        margin=dict(l=60, r=20, t=70, b=50),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0, font=dict(size=11)),
    )
    fig.update_xaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    fig.update_yaxes(showgrid=True, gridcolor="rgba(0,0,0,0.07)", zeroline=True)
    return fig

In [2]:
@dataclass(frozen=True)
class Design:
    """Параметры теста непрерывной метрики (средний чек)."""
    alpha: float = 0.05        # односторонняя ошибка I рода
    beta: float = 0.20         # ошибка II рода (power = 0.80)
    sigma: float = 100.0       # СКО метрики, руб.
    mde: float = 10.0          # абсолютный MDE, руб.
    K: int = 5
    t: tuple = (0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0)  # доли информации (моменты взглядов)

    @property
    def delta(self) -> float:
        return self.mde / self.sigma

    @property
    def theta_mde(self) -> float:
        return norm.ppf(1 - self.alpha) + norm.ppf(1 - self.beta)

    @property
    def i_max(self) -> float:
        return (self.theta_mde / self.delta) ** 2

    @property
    def n_max(self) -> float:
        return 2 * self.sigma ** 2 * self.i_max

design = Design()
design

Design(alpha=0.05, beta=0.2, sigma=100.0, mde=10.0, K=5, t=(0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0))

## 0. Постановка задачи


Есть две разновидности ранней остановки эксперимента:

- **Остановка по эффективности (efficacy)** — эффект статистически доказан до оконания эксперимента. Останавливаемся, потому что *победили*.
- **Остановка по бесперспективности (futility)** — эффекта, скорее всего, нет или он меньше MDE. Останавливаемся, чтобы не жечь трафик на заведомо «мёртвый» тест.

**Остановка по бесперспективности (futility)** появляется ка кестественный ответ на следующий вопрос:
> *Если на промежуточном взгляде эффект слабее MDE (или его нет вовсе), с какой вероятностью тест всё-таки дойдёт до значимости к концу?*

Если эта вероятность мала — тратить оставшийся трафик бессмысленно, и можно завершить тест.

## 1. Математическая разработка подхода

### 1. $Z$  и  $B$  значения

<details>
<summary>1.1. Вывод распределения $Z_t$</summary>

Пусть у нас есть группы $A$ (контроль) и $B$ (тест), по $n_t$ i.i.d. наблюдений в каждой с общей дисперсией $\sigma^2$. В качестве эффекта мы будем оценивать разность средних: $\hat\delta_t=\bar X_B-\bar X_A$.

Дисперсия оцениваемой разницы среднего будет $\mathrm{Var}(\hat\delta_t)=2\sigma^2/n_t$, а $SE_t=\sigma\sqrt{2/n_t}$
1. $\mathrm{Var}(\bar X_A)=\mathrm{Var}\!\left(\frac1{n_t}\sum X_{A,i}\right)=\frac{1}{n_t^2}\sum\mathrm{Var}(X_{A,i})=\frac{1}{n_t^2}\cdot n_t\sigma^2=\frac{\sigma^2}{n_t}$.
2. Аналогично п.1 для $B$.
3. Так как выборки независимы, дисперсии складываются: $\mathrm{Var}(\hat\delta_t)= \mathrm{Var}(\bar X_A) + \mathrm{Var}(\bar X_B)$ = $2\sigma^2/n_t$
4. $SE_t=\sqrt{\mathrm{Var}(\hat\delta_t)} = \sigma\sqrt{2/n_t}$.

После стандартизации $\hat\delta_t$ получаемая величину $Z_t :=\hat\delta_t/SE_t$, которая распределена как $\mathcal N\big(\delta/SE_t,\,1\big)$.

Введём **информацию** $I_t$ - величину обратную дисперсии оценки; $I_t =1/SE_t^2=n_t/(2\sigma^2)$. Через $I_t$ обозначим момент времени $t$, который показывает какую часть информации мы увидели: $t=I_t/I_{max}\approx n_t/n_{max}$.

Через информацию таже обозначим и **дрейф** $\theta$ - наблюдаемый эффект, отмасштабированный на дисперсию: $\theta=\delta\sqrt{I_{max}}$.


Через эти 2 новые величины переобозначим математическое ожидание $Z_t$  для более компактной записи, которой будем пидерживаться дальше все время:

$$\mathbb E[Z_t]=\frac{\delta}{SE_t}=\delta\sqrt{I_t}=\underbrace{\delta\sqrt{I_{max}}}_{=:\theta}\cdot\underbrace{\sqrt{I_t/I_{max}}}_{=\sqrt t}=\theta\sqrt t.$$

$$\boxed{Z_t\sim\mathcal N(\theta\sqrt t,\,1)}$$

</details>

<details>
<summary>1.2. Независимые приращения и ковариация</summary>

<details>
<summary>Факт 1. Оценка в момент $t$ раскладывается на старый и новый куски $$Z_t=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}.$$</summary>

  Новые данные не знают о старых. Данные приходят порциями, и оценка в момент $t$ — это взвешенное среднее старой оценки и оценки только по новым данным. Распишем среднее по $n_t$ наблюдениям как сумму по первым $n_s$ и по остальным $n_t-n_s$:

$$\bar X_B^{(t)}=\frac1{n_t}\sum_{i=1}^{n_t}X_{B,i}=\frac1{n_t}\left(\sum_{i=1}^{n_s}X_{B,i}+\sum_{i=n_s+1}^{n_t}X_{B,i}\right)=\frac{n_s}{n_t}\bar X_B^{(s)}+\frac{n_t-n_s}{n_t}\bar X_B^{(t-s)}.$$

То же для $A$, вычитаем:

$$\hat\delta_t=\frac{n_s}{n_t}\hat\delta_s+\frac{n_t-n_s}{n_t}\hat\delta_{t-s}.$$

Делим обе части на $SE_t$:

$$Z_t=\frac{n_s}{n_t}\cdot\frac{\hat\delta_s}{SE_t}+\frac{n_t-n_s}{n_t}\cdot\frac{\hat\delta_{t-s}}{SE_t}.$$

Каждый кусок пересчитывается в свою нормированную статистику:

$$
\frac{\hat\delta_s}{SE_t}=\frac{\hat\delta_s}{SE_s}\cdot\frac{SE_s}{SE_t}=Z_s\cdot\frac{SE_s}{SE_t} = Z_s\cdot\frac{\sigma\sqrt{2/n_s}}{\sigma\sqrt{2/n_t}}=Z_s\cdot\sqrt{\frac{n_t}{n_s}} \\
\frac{\hat\delta_{t-s}}{SE_t}=\frac{\hat\delta_{t-s}}{SE_{t-s}}\cdot\frac{SE_{t-s}}{SE_t}=Z_{t-s}\cdot\frac{SE_{t-s}}{SE_t} = Z_{t-s}\cdot\frac{\sigma\sqrt{2/n_{t-s}}}{\sigma\sqrt{2/n_t}}=Z_{t-s}\cdot\sqrt{\frac{n_t}{n_{t-s}}}=Z_{t-s}\cdot\sqrt{\frac{n_t}{n_t-n_s}}
$$

Тогда

$$Z_t=\frac{n_s}{n_t} \cdot \sqrt{\frac{n_t}{n_s}} \cdot Z_s + \frac{n_t-n_s}{n_t} \cdot \sqrt{\frac{n_t}{n_t-n_s}} \cdot Z_{t-s} \\
Z_t=\sqrt{\frac{n_s}{n_t}} \cdot Z_s+\sqrt{\frac{n_{t-s}}{n_t}} \cdot Z_{t-s}=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}
$$    
</details>

<details>
<summary>Факт 2. Ковариация и корреляция $Z$-величин двух периодов пропорционально корню отношения моментов $$\mathrm{Cov}(Z_s,Z_t)=\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$$</summary>

Для доказательства используем билинеейность ковариации:
$$\mathrm{Cov}(aX+bY,Z)=a \cdot \mathrm{Cov}(X,Z)+b \cdot \mathrm{Cov}(Y,Z)$$

Берём разложение $Z_t=\sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}$,и подставим в формулу:

$$\mathrm{Cov}(Z_s,Z_t)=\mathrm{Cov}\Big(Z_s, \sqrt{\frac{s}{t}} \cdot Z_s+\sqrt{\frac{t-s}{t}} \cdot Z_{t-s}\Big)$$

По билинейности разваливается на два слагаемых:

$$=\sqrt{\frac{s}{t}} \cdot \underbrace{\mathrm{Cov}(Z_s,Z_s)}_{=1} + \sqrt{\frac{t-s}{t}} \cdot \underbrace{\mathrm{Cov}(Z_{t-s},Z_s)}_{=0}=\sqrt{\frac{s}{t}}.$$

Так как $\mathrm{Var}(Z_s)=\mathrm{Var}(Z_t)=1$, то и
$$\mathrm{corr}(Z_s,Z_t)=\sqrt{s/t}$$
</details>
</details>

<details>
<summary>1.3. Переход к $B(t)$</summary>

$Z_t$ прекрасен тем, что безразмерен и всегда $\mathcal N(\theta\sqrt t,1)$. Но у него есть две неприятности, которые мешают строить границы.

**Дефект 1**: $Z$ не «накапливается». Дисперсия всегда 1, значит прирост дисперсии нулевой:

$$\mathrm{Var}(Z_t)-\mathrm{Var}(Z_s)=1-1=0.$$

А само приращение $Z_t-Z_s$ имеет дисперсию

$$\mathrm{Var}(Z_t-Z_s)=1+1-2\sqrt{s/t},$$

которая зависит от обоих моментов и не равна длине интервала, т.е. «сколько нового пришло» в единицах $Z$ не измеряется.

**Дефект 2**: в разложении появляются веса при $Z_s$ и $Z_{t-s}$, которые надо таскать.

<br>

Обе проблемы — от одного корня: мы поделили на $SE_t$, а $SE_t\propto1/\sqrt t$ меняется со временем. Чтобы избавиться от переменного делителя домножим на $\sqrt t$  и будем вести расчёты в новой величине $B_t= \sqrt t \cdot Z_t = \sqrt t\cdot\frac{\hat\delta_t}{SE_t}$.
У новой величины получаются следующие моменты и величины:
1. $\mathbb E[B(t)]=\sqrt t\cdot\theta\sqrt t=\theta t$
2. $\mathrm{Var}[B(t)]=t\cdot1=t$
3. $\mathrm{Cov}\big(B(s),B(t)\big)=\mathrm{Cov}\big(\sqrt s \cdot Z_s,\sqrt t \cdot Z_t\big)=\sqrt{st}\cdot\underbrace{\mathrm{Cov}(Z_s,Z_t)}_{=\sqrt{s/t}}=\sqrt{st}\cdot\sqrt{\frac{s}{t}}=s$
4. $\mathrm{Cov}\big(B(s),B(t)-B(s)\big)=\mathrm{Cov}(B(s),B(t))-\mathrm{Var}(B(s))=s-s=0$

<br>

На шкале $B$ разложение — простое сложение без весов:

$$B(t)=\sqrt t\left(\sqrt{\frac{s}{t}}Z_s+\sqrt{\frac{t-s}{t}}Z_{t-s}\right)=\sqrt s \cdot Z_s+\sqrt{t-s} \cdot Z_{t-s}=B(s)+\Delta B$$


Так как $(B(s),B(t))$ совместно нормальны (это линейные комбинации независимых нормальных приращений), нулевая ковариация означает независимость:

$$B(t)-B(s)\perp B(s) , \qquad
B(t)-B(s)\sim\mathcal N(\theta(t-s),t-s)$$

<br>

Величина $B$ так же имеет и содержательное ядро:

Подставим $Z_t=\hat\delta_t\sqrt{I_t}$ (так как $SE_t=1/\sqrt{I_t}$):

$$B(t)=\sqrt t,Z_t=\sqrt{\frac{I_t}{I_{max}}}\cdot\hat\delta_t\sqrt{I_t}=\frac{\hat\delta_t I_t}{\sqrt{I_{max}}}.$$

Раскроем $\hat\delta_t I_t$ через данные:

$$\hat\delta_t I_t=\left(\frac1{n_t}\sum_{i=1}^{n_t}\Delta_i\right)\frac{n_t}{2\sigma^2}=\frac{1}{2\sigma^2}\sum_{i=1}^{n_t}\Delta_i,\qquad \Delta_i=X_{B,i}-X_{A,i}.$$

Итого

$$\boxed{B(t)=\frac{1}{2\sigma^2\sqrt{I_{max}}}\sum_{i=1}^{n_t}\big(X_{B,i}-X_{A,i}\big)}$$

$B$ — с точностью до константы простая накопленная сумма по пользователям. Получается, что прирост $B(t)-B(s)$ - сумма по новым пользователям, независимая от суммы по старым. Формально $B$ — броуновское движение с дрейфом $\theta$.
<br>

**Переход границ между шкалами.** Правило на $Z$-шкале $Z_k>c_k$ домножением на $\sqrt{t_k}$ превращается в

$$Z_k>c_k\iff B(t_k)>c_k\sqrt{t_k}=:b_k.$$

Вся дальнейшая техника работает на шкале $B$, а результаты возвращаются на $Z$ делением на $\sqrt{t_k}$.
</details>

In [3]:
def z_stat(xA: np.ndarray, xB: np.ndarray, sigma: float) -> float:
    """Двухвыборочный z-тест непрерывной метрики (известная дисперсия)."""
    nA, nB = xA.size, xB.size
    se = sigma * np.sqrt(1 / nA + 1 / nB)
    return (xB.mean() - xA.mean()) / se


def simulate_b(rng, theta: float, t: np.ndarray, n_sim: int = 200) -> np.ndarray:
    """Траектории B(t) = sqrt(t)*Z_t — броуновское движение с дрейфом theta."""
    dt = np.diff(np.concatenate(([0.0], t)))
    inc = rng.normal(loc=theta * dt, scale=np.sqrt(dt), size=(n_sim, len(t)))
    return np.cumsum(inc, axis=1)


In [4]:
t = np.asarray(design.t)

modes = [
    (0.0,              "H<sub>0</sub> (θ = 0)",    C_CONT),
    (design.theta_mde, "H<sub>1</sub> (θ = MDE)",  C_EFF),
]

B_by_mode = [(theta, name, color, simulate_b(rng, theta, t, 250))
             for theta, name, color in modes]

x_ticks = np.linspace(t.min(), t.max(), 20)
y_all   = np.concatenate([B.ravel() for *_, B in B_by_mode])
y_lo    = min(y_all.min(), 0.0)
y_hi    = max(y_all.max(), modes[1][0] * t.max())
y_ticks = np.linspace(y_lo, y_hi, 20)

# Подписи панелей — с расшифровкой смысла, чтобы легенда была не нужна
subplot_titles = (
    "H<sub>0</sub> (θ = 0)",
    "H<sub>1</sub> (θ = MDE)",
)

fig = make_subplots(
    rows=1, cols=2, shared_yaxes=True,
    horizontal_spacing=0.06,
    subplot_titles=subplot_titles,
)

for col, (theta, name, color, B) in enumerate(B_by_mode, start=1):
    for i in range(B.shape[0]):
        fig.add_trace(
            go.Scatter(
                x=t, y=B[i], mode="lines",
                line=dict(color=color, width=0.8),
                opacity=0.25,
                showlegend=False,
                hoverinfo="skip",
            ),
            row=1, col=col,
        )
    # Чёрная штриховая линия дрейфа (легенды нет — подпишем аннотацией ниже)
    fig.add_trace(
        go.Scatter(
            x=t, y=theta * t, mode="lines",
            line=dict(color="black", width=2.5, dash="dash"),
            showlegend=False,
            name=f"E[B(t)] = {theta:.2f}·t",
        ),
        row=1, col=col,
    )

    # Небольшая подпись у линии дрейфа — вместо легенды
    fig.add_annotation(
        text=f"E[B(t)] = {theta:.2f}·t",
        x=t.max(), y=theta * t.max(),
        xanchor="right", yanchor="bottom",
        showarrow=False,
        font=dict(size=11, color="black"),
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="rgba(0,0,0,0.35)", borderwidth=1,
        row=1, col=col,
    )

fig.update_xaxes(
    tickmode="array", tickvals=x_ticks, tickformat=".2f",
    title_text="Доля информации t", automargin=True,
)
fig.update_yaxes(
    tickmode="array", tickvals=y_ticks, tickformat=".2f",
    automargin=True,
)
fig.update_yaxes(title_text="B(t)", col=1)

fig.update_layout(
    autosize=True,
    height=720,
    margin=dict(t=70, b=80, l=70, r=40),
    title=dict(
        text="Броуновское движение при H<sub>0</sub> и H<sub>1</sub>",
        x=0.5, xanchor="center",
        y=0.98, yanchor="top", yref="container",
        font=dict(size=16),
    ),
    showlegend=False,
)

fig

### 2. Conditional Power (CP)

<details>
<summary>1. Условная вероятность «добить» тест</summary>

$\mathrm{CP}(\theta)=P_\theta(Z_1>z_{1-\alpha}\mid Z_t=z_t)$ — вероятность дойти до значимости при заданном дрейфе $\theta$ и текущих данных. Это и есть ответ на вопрос «стоит ли тратить оставшийся трафик».


Выразим финальную статистику через текущую и приращение:
$$Z_1=B(1)=B(t)+\big(B(1)-B(t)\big)=\sqrt t \cdot z_t+\big(B(1)-B(t)\big).$$

Приращение независимо от $B(t)$ и распределено как $\mathcal N(\theta(1-t),1-t)$. Поэтому при условии $Z_t=z_t$ первое слагаемое — детерминированное число $\sqrt t \cdot z_t$, а второе — случайное. Получается, что финальная статистика имеет условное распределение:
$$(Z_1 \mid Z_t=z_t) \sim \mathcal N\big(\sqrt t \cdot z_t+\theta(1-t);1-t\big)$$

<br>
Вероятность попасть в критическую область ${Z_1>z_{1-\alpha}}$ — хвост этого распределения.
Стандартизуем: $\mu=\sqrt t z_t+\theta(1-t)$, $\sigma=\sqrt{1-t}$:
$$\mathrm{CP}(\theta)=P \left(\frac{Z_1-\mu}{\sigma}>\frac{z_{1-\alpha}-\mu}{\sigma}\right)=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)}{\sqrt{1-t}}\right)$$

$$\boxed{\mathrm{CP}(\theta)=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)}{\sqrt{1-t}}\right)}$$


Предельные случаи.

* $t\to0$: $\sqrt t z_t\to0$, $\theta(1-t)\to\theta$, $\sqrt{1-t}\to1$, поэтому $\mathrm{CP}\to1-\Phi(z_{1-\alpha}-\theta)$ — априорная мощность дизайна.
* $t\to1$: знаменатель $\sqrt{1-t}\to0$. При $z_t<z_{1-\alpha}$ числитель $\to z_{1-\alpha}-z_t>0$, то есть $u\to+\infty$, $\Phi\to1$, $\mathrm{CP}\to0$. При $z_t>z_{1-\alpha}$ числитель $\to$ отрицателен, $\mathrm{CP}\to1$. Получается ступенька в точке $z_t=z_{1-\alpha}$.
Выбор $\theta$ (раздел 2.2 лекции).
</details>

<details>
<summary>2. Монотонность по $z_t$</summary>

Обозначим $u=(z_{1-\alpha}-\sqrt t,z_t-\theta(1-t))/\sqrt{1-t}$. Тогда

$$\frac{d,\mathrm{CP}}{dz_t}=-\Phi'(u)\cdot\frac{du}{dz_t}=-\varphi(u)\cdot\left(-\frac{\sqrt t}{\sqrt{1-t}}\right)=\varphi(u)\sqrt{\frac{t}{1-t}}>0.$$

Поскольку плотность $\varphi(u)>0$, CP строго возрастает по $z_t$ — значит, уравнение $\mathrm{CP}=\gamma$ имеет единственный корень.
</details>


<details>
<summary>3. Какое $\theta$ подставить</summary>
Какое $\theta$ подставить — самый тонкий момент. Три осмысленных варианта:

1. $\theta=0$ (под $H_0$): «даже если эффекта нет, дойдём ли до ложной значимости?» — консервативно, редко останавливает.
2. $\theta=\hat\theta_t=z_t/\sqrt t$ (текущий тренд): «будущее похоже на то, что уже видим» — заманчиво, но оптимистично: ранняя оценка шумная, и случайно «выстреливший» эффект кажется многообещающим.
3. $\theta=\theta_{MDE}$ (дизайновый эффект): «если эффект ровно такой, как закладывали в power-расчёте» — стандартный ориентир, отвечает на вопрос «есть ли шанс поймать то, ради чего затевался тест».
</details>


Правило futility: **если $\mathrm{CP} < \gamma$ — останавливаемся** (типично $\gamma \in [0.10, 0.20]$).


#### 2.1. Построение CP

In [5]:
def conditional_power(z_t, t, theta, z_alpha=1.6448536):
    """CP: вероятность Z_1 > z_alpha при наблюдаемом z_t и истинном дрейфе theta."""
    z_t = np.asarray(z_t, dtype=float)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - theta * (1 - t)) / np.sqrt(1 - t))


def ppos(z_t, t, theta0, tau, z_alpha=1.6448536):
    """Predictive probability of success (нормальный prior, замкнутая форма)."""
    z_t = np.asarray(z_t, dtype=float)
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)
    return 1.0 - norm.cdf((z_alpha - np.sqrt(t) * z_t - mu * (1 - t))
                          / np.sqrt((1 - t) + (1 - t)**2 * sigma2))


In [6]:
z_alpha = norm.ppf(1 - design.alpha)
z_grid = np.linspace(-3, 3, 300)
tt = 0.5

fig = go.Figure()
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, 0.0, z_alpha),
                         mode="lines", line=dict(color=C_CONT, width=2.5), name="CP под H0"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, design.theta_mde, z_alpha),
                         mode="lines", line=dict(color=C_EFF, width=2.5), name="CP под MDE"))
fig.add_trace(go.Scatter(x=z_grid, y=conditional_power(z_grid, tt, z_grid / np.sqrt(tt), z_alpha),
                         mode="lines", line=dict(color=C_FUT, width=2.5), name="CP под трендом"))
fig.add_hline(y=0.15, line_dash="dash", line_color="black", annotation_text="порог γ = 0.15")
fig = style(fig, title=f"CP при t = {tt}: выбор θ кардинально меняет ответ",
            xlabel="наблюдаемая $z_t$", ylabel="Conditional Power")
fig


#### 2.2. Карта CP

Порог $\mathrm{CP}<\gamma$ перепишем в терминах статистики. При фиксированных $t,\alpha,\gamma,\theta$ определим $a_{CP}$ как решение

$$\mathrm{CP}(a_{CP};t,\theta)=\gamma.$$

Так как CP монотонно растёт по $z_t$ (показано в §2), это уравнение имеет единственный корень, и

$$\mathrm{CP}(z_t)<\gamma\iff z_t<a_{CP}(t).$$

Численно корень ищут методом деления пополам или Брента (в лекции — brentq). На плоскости $(t,z)$ правило «стоп по CP» — кривая $a_{CP}(t)$, ниже которой останавливаемся.

Карта CP на плоскости $(t, z)$ показывает, почему время взгляда так важно: на ранних взглядах карта почти плоская (судьбу решает будущее), к концу «заостряется» в ступеньку. Красный контур $\mathrm{CP}=\gamma$ — граница «стоп / продолжаем».

In [18]:
from functools import lru_cache
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm
from scipy.optimize import brentq
from ipywidgets import interact, FloatSlider

Z_ALPHA = 1.6448536


@lru_cache(maxsize=4096)
def _cp_boundary_cached(t_round, gamma, theta, z_alpha=Z_ALPHA):
    f = lambda z: conditional_power(z, t_round, theta, z_alpha) - gamma
    try:
        return brentq(f, -8, 8)
    except ValueError:
        return np.nan


def cp_to_boundary(t, gamma, theta, z_alpha=Z_ALPHA):
    return _cp_boundary_cached(round(float(t), 4), float(gamma), float(theta), z_alpha)


def plot_cp_heatmap(theta=design.theta_mde, gamma=0.15):
    t_grid = np.linspace(0.05, 0.95, 120)
    z_grid = np.linspace(-5, 5, 240)
    T, Z = np.meshgrid(t_grid, z_grid)
    CP = conditional_power(Z, T, theta, Z_ALPHA)

    a_cp = np.array([cp_to_boundary(t_, gamma, theta, Z_ALPHA) for t_ in t_grid])

    fig = go.Figure()
    fig.add_trace(go.Heatmap(
        z=CP, x=t_grid, y=z_grid,
        colorscale="Viridis",
        colorbar=dict(title="CP"),
        showscale=True,
        zmin=0, zmax=1,
    ))
    fig.add_trace(go.Scatter(
        x=t_grid, y=a_cp, mode="lines",
        line=dict(color="red", width=3),
        name=f"CP = {gamma:.2f}",
    ))

    fig.update_layout(
        autosize=True,
        height=520,
        margin=dict(l=70, r=20, t=70, b=60),
        uirevision="cp_heatmap",
        # HTML-разметка вместо MathJax — рендерится всегда
        title=dict(
            text=(f"Conditional power под MDE: контур — граница futility "
                  f"(θ={theta:.2f}, γ={gamma:.2f})"),
            x=0.5, xanchor="center",
        ),
        xaxis=dict(
            title="Доля информации <i>t</i>",
            dtick=0.05,
            tickformat=".2f",
            showgrid=True,
            zeroline=False,
        ),
        yaxis=dict(
            title="<i>z</i><sub>t</sub>",
            dtick=1,
            tickformat=".0f",
            showgrid=True,
            zeroline=False,
        ),
    )
    fig.update_xaxes(range=[t_grid[0], t_grid[-1]])
    fig.update_yaxes(range=[z_grid[0], z_grid[-1]])

    fw = go.FigureWidget(fig)
    fw.layout.autosize = True
    return fw


interact(
    plot_cp_heatmap,
    theta=FloatSlider(value=design.theta_mde, min=0, max=4, step=0.1,
                      description="θ"),
    gamma=FloatSlider(value=0.15, min=0.05, max=0.50, step=0.05,
                      description="γ"),
)

interactive(children=(FloatSlider(value=2.4864748605243863, description='θ', max=4.0), FloatSlider(value=0.15,…

<function __main__.plot_cp_heatmap(theta=np.float64(2.4864748605243863), gamma=0.15)>

#### 2.3. Особенность CP

CP-граница строится из условного расчёта на каждом взгляде (подставляем наблюдаемое $z_t$) и интерпретируется как «вероятность успеха», но она не гарантирует заранее, сколько ошибки II рода будет «съедено» по всем траекториям. Альтернатива — зафиксировать границу до данных с гарантией контроля ошибки II рода: β-spending (будет дальше)

### 3. Predictive Power (PPoS)
<details>
<summary>1. Определение</summary>
$$\mathrm{PPoS}=\int \mathrm{CP}(\theta),\pi(\theta\mid D_t),d\theta.$$

Вместо «шансы при одном фиксированном θ» — «средние шансы по всем правдоподобным θ, взвешенным апостериорной вероятностью»

</details>

<details>
<summary>2. Точность апостериора $\pi(\theta)$ </summary>

Априор $\theta\sim\mathcal N(\theta_0,\tau^2)$. Правдоподобие $Z_t\mid\theta\sim\mathcal N(\theta\sqrt t,1)$ эквивалентно наблюдению $y=z_t/\sqrt t$ с шумом $y\mid\theta\sim\mathcal N(\theta,1/t)$. Плотности:

$$\pi(\theta)\propto\exp\left(-\frac{(\theta-\theta_0)^2}{2\tau^2}\right),\qquad L(\theta)\propto\exp\left(-\frac{t(\theta-y)^2}{2}\right)$$

Апостериор $\propto\pi L$; логарифмируем и раскрываем квадраты:

$$-2\log = \frac{(\theta-\theta_0)^2}{\tau^2}+t(\theta-y)^2 = \underbrace{\left(\frac1{\tau^2}+t\right)}_{=:A}\theta^2-2\underbrace{\left(\frac{\theta_0}{\tau^2}+ty\right)}_{=:B}\theta+\text{const}$$

Дополняем до полного квадрата: $A\theta^2-2B\theta=A(\theta-B/A)^2+\text{const}$. Значит апостериор нормален с точностью $A$ и средним $B/A$:

$$\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+ty\right)$$

Подставим $ty=t\cdot(z_t/\sqrt t)=\sqrt t z_t$:

$$\boxed{\frac1{\sigma_p^2}=\frac1{\tau^2}+t,\qquad \mu_p=\sigma_p^2\left(\frac{\theta_0}{\tau^2}+\sqrt t,z_t\right)}$$

* Точность $\pi(\theta)$ = точность prior + точность данных.
* $\mu_p$ — взвешенная сумма prior и данных с весами-точностями.
</details>

<details>
<summary>3. Замкнутость PPoS </summary>

Числитель CP линеен по $\theta$. Перепишем с $\theta=\mu_p+W$, где $W\sim\mathcal N(0,\sigma_p^2)$:

$$z_{1-\alpha}-\sqrt t \cdot z_t-\theta(1-t)=z_{1-\alpha}-\sqrt t \cdot z_t-\mu_p(1-t)-W(1-t)=:A-(1-t)W.$$

Обозначим $\widetilde W=(1-t)W\sim\mathcal N\big(0,(1-t)^2\sigma_p^2\big)$, тогда

$$\mathrm{CP}(\theta)=1-\Phi\left(\frac{A-\widetilde W}{\sqrt{1-t}}\right)$$

Для $W\sim\mathcal N(0,\sigma_W^2)$ и независимого $U\sim\mathcal N(0,1)$:

$$\Phi\left(\frac{a-W}{b}\right)=P\left(U\le\frac{a-W}{b}\right).$$

Беря матожидание по $W$ (теорема полной вероятности):

$$\mathbb E_W\left[\Phi\left(\frac{a-W}{b}\right)\right]=P\left(U\le\frac{a-W}{b}\right)=P(bU+W\le a)=\Phi\left(\frac{a}{\sqrt{b^2+\sigma_W^2}}\right),$$

так как $bU+W\sim\mathcal N(0,,b^2+\sigma_W^2)$.

Здесь $a=A$, $b=\sqrt{1-t}$, $\sigma_W^2=(1-t)^2\sigma_p^2$:

$$\boxed{\mathrm{PPoS}=1-\Phi\left(\frac{z_{1-\alpha}-\sqrt t,z_t-\mu_p(1-t)}{\sqrt{(1-t)+(1-t)^2\sigma_p^2}}\right)}$$
</details>

In [24]:
def plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8):
    """PPoS = E_π[CP(θ)], апостериор и CP(θ) на одном полотне."""
    # --- апостериор (замкнутая форма, нормальный prior) ---
    sigma2 = 1.0 / (1.0 / tau**2 + t)
    mu = sigma2 * (theta0 / tau**2 + np.sqrt(t) * z_t)

    theta_grid = np.linspace(-0.5, 4, 400)
    post = norm.pdf(theta_grid, loc=mu, scale=np.sqrt(sigma2))
    cp_theta = conditional_power(z_t, t, theta_grid, Z_ALPHA)
    pp = ppos(z_t, t, theta0, tau, Z_ALPHA)
    cp_trend = conditional_power(z_t, t, z_t / np.sqrt(t), Z_ALPHA)

    # --- нормируем шкалы, чтобы плотность и вероятность читались вместе ---
    post_scaled = post / max(post.max(), 1e-12) * 0.9
    cp_weighted = cp_theta * post
    cp_weighted_scaled = cp_weighted / max(cp_weighted.max(), 1e-12) * 0.9

    fig = go.Figure()

    # апостериор (нормированная плотность, заливка)
    fig.add_trace(go.Scatter(
        x=theta_grid, y=post_scaled, mode="lines",
        line=dict(color="rgba(120,120,120,0.9)", width=2.5),
        fill="tozeroy", fillcolor="rgba(153,153,153,0.15)",
        name="апостериор π(θ|данные) (норм.)",
    ))

    # CP(θ)
    fig.add_trace(go.Scatter(
        x=theta_grid, y=cp_theta, mode="lines",
        line=dict(color="rgba(31,119,180,1)", width=2.5),
        name="CP(θ)",
    ))

    # CP(θ)·π(θ) — нормированная заливка
    fig.add_trace(go.Scatter(
        x=theta_grid, y=cp_weighted_scaled, mode="lines",
        line=dict(color="rgba(217,95,2,0.9)", width=2),
        fill="tozeroy", fillcolor="rgba(217,95,2,0.25)",
        name="CP(θ)·π(θ) (норм.)",
    ))

    # вертикальные референсы
    fig.add_vline(
        x=mu, line_dash="dash", line_color="black",
        annotation_text=f"μ = {mu:.2f}",
        annotation_position="top",
    )
    fig.add_vline(
        x=design.theta_mde, line_dash="dot", line_color="rgba(31,119,180,1)",
        annotation_text=f"θ_MDE = {design.theta_mde:.2f}",
        annotation_position="bottom",
    )

    # горизонтальные референсы
    fig.add_hline(
        y=pp, line_dash="dash", line_color="rgba(217,95,2,1)",
        annotation_text=f"PPoS = {pp:.3f}",
        annotation_position="right",
    )
    fig.add_hline(
        y=cp_trend, line_dash="dash", line_color="rgba(120,120,120,1)",
        annotation_text=f"CP(тренд) = {cp_trend:.3f}",
        annotation_position="right",
    )

    fig.update_layout(
        autosize=True,
        height=480,
        margin=dict(l=70, r=90, t=70, b=60),
        uirevision="ppos_plot",
        title=dict(
            text=(f"PPoS = E_π[CP(θ)] = {pp:.3f}  "
                  f"(z_t={z_t:.2f}, t={t:.2f}, θ₀={theta0:.2f}, τ={tau:.2f})"),
            x=0.5, xanchor="center",
        ),
        xaxis=dict(
            title="θ",
            dtick=0.5,
            tickformat=".1f",
            showgrid=True,
            zeroline=False,
            range=[theta_grid[0], theta_grid[-1]],
        ),
        yaxis=dict(
            title="плотность / вероятность",
            dtick=0.1,
            tickformat=".1f",
            showgrid=True,
            zeroline=False,
            range=[0, 1.0],
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom", y=1.02,
            xanchor="center", x=0.5,
            bgcolor="rgba(255,255,255,0.7)",
        ),
        hovermode="x unified",
    )

    return go.FigureWidget(fig)


interact(
    plot_ppos,
    z_t=FloatSlider(value=0.4, min=-3, max=3, step=0.05, description="z_t"),
    t=FloatSlider(value=0.5, min=0.05, max=0.95, step=0.05, description="t"),
    theta0=FloatSlider(value=0.0, min=-1, max=3, step=0.1, description="prior θ0"),
    tau=FloatSlider(value=0.8, min=0.05, max=3, step=0.05, description="prior τ"),
)

interactive(children=(FloatSlider(value=0.4, description='z_t', max=3.0, min=-3.0, step=0.05), FloatSlider(val…

<function __main__.plot_ppos(z_t=0.4, t=0.5, theta0=0.0, tau=0.8)>

### 4. β-spending

#### 4.1. Предварительная фиксация futility-границы.

<details>
<summary>Логика симметрии с α-spending</summary>

* α-spending распределяет бюджет ошибки I рода по взглядам под $H_0$;
* β-spending — то же для ошибки II рода: «распределить β так, чтобы суммарная вероятность зря остановить тест при реальном эффекте была не больше β».

«Зря остановить» возможно только когда эффект есть, поэтому futility-границу считают под $H_1$ ($\theta=\theta_{MDE}$), а efficacy — под $H_0$.

Спендинг-функции — неубывающие функции доли информации $t$: $\alpha(0)=\beta(0)=0$, $\alpha(1)=\alpha$, $\beta(1)=\beta$.
В примерах ниже будет использовать степенное семейство $\beta(t)=\beta t^{\rho}$: $\rho=3$ — O' Brien-Fleming -подобная (консервативно рано), $\rho=1$ — Pocock-подобная (агрессивно рано).

</details>

<details>
<summary>Рекурсивный алгоритм расчёта</summary>

Введем следующие обозначения
* приращения бюджетов ошибок:

$$\Delta\alpha_k=\alpha(t_k)-\alpha(t_{k-1}),\qquad \Delta\beta_k=\beta(t_k)-\beta(t_{k-1}).$$

* $p_k(x)$ — (суб)плотность $B(t_k)$ среди траекторий, которые «дожили» до взгляда $k-1$, не пересекнув ни одну границу. То есть $\int p_k(x)dx$ — вероятность ещё не остановиться к этому моменту.

<br>

Алгоритм состоит из 3 шагов и строит рекурсивно границу

1. **Шаг 1. Инициализация ($k=1$, приращение от 0 до $t_1$)**
Первый раз считаем нормальную плотность под нужной гипотезы:

    * Для efficacy под $H_0$: $p_1^{eff}(x)=\varphi(x,0,t_1)$;
    * Для futility под $H_1$: $p_1^{fut}(x)=\varphi(x,\theta t_1,t_1)$.


2. **Шаг 2. Отсечения (взгляд $k$)**

    * Efficacy-граница $b_k$ отсекает правый хвост с массой $\Delta\alpha_k$
    * Futility-граница $a_k$ отсекает левую часть с массой $\Delta\beta_k$

$$\int_{b_k}^{\infty}p_k^{eff}(x)dx=\Delta\alpha_k$$
$$\int_{-\infty}^{a_k}p_k^{fut}(x)dx=\Delta\beta_k$$

3. **Шаг 3. Свёртка**

    Оставляем только траектории в области продолжения и добавляем независимое приращение $B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta_{k+1},\Delta_{k+1})$, $\Delta_{k+1}=t_{k+1}-t_k$:

$$p_{k+1}^{fut}(y)=\int_{a_k}^{b_k}p_k^{fut}(x)\varphi\big(y,x+\theta\Delta_{k+1},\Delta_{k+1}\big)dx$$
$$p_{k+1}^{eff}(y)=\int_{a_k}^{b_k}p_k^{eff}(x)\varphi\big(y,x+\theta\Delta_{k+1},\Delta_{k+1}\big)dx$$

4. После построения границ их необходимо перевести обратно в Z шкалу:
$$c_k=\frac{b_k}{\sqrt{t_k}},\qquad a_k^{(\text{Z})}=\frac{a_k}{\sqrt{t_k}}.$$

Почему это работает. Урезание оставляет только «ещё продолжающиеся» траектории, а свёртка прибавляет следующий независимый шаг. Независимость приращений (доказана в §1.3) гарантирует, что распределение следующего шага зависит только от текущей позиции, а не от всей истории, — поэтому рекурсия точна (с точностью до дискретизации сетки $x$). Каждый шаг «тратит» ровно приращение бюджета $\Delta\alpha_k$ или $\Delta\beta_k$; суммируясь по $k$, они дают в точности $\alpha$ и $\beta$.

</details>

Введем следующие обозначения
* приращения бюджетов ошибок:

$$\Delta\alpha_k=\alpha(t_k)-\alpha(t_{k-1}),\qquad \Delta\beta_k=\beta(t_k)-\beta(t_{k-1}).$$

* $p_k(x)$ — (суб)плотность $B(t_k)$ среди траекторий, которые «дожили» до взгляда $k-1$, не пересекнув ни одну границу. То есть $\int p_k(x)dx$ — вероятность ещё не остановиться к этому моменту.

<br>

Алгоритм состоит из 3 шагов и строит рекурсивно границу

**Шаг 1. Инициализация (считаем безусловную плотность $B(t_1)$)**
  
  $p_1(x)$ — это плотность положения процесса $B(t_1)$ в момент первого взгляда, при условии, что до этого момента мы никого не отбрасывали. А раз никого не отбрасывали — это просто безусловное распределение $B(t_1)$, и вся масса на месте: $\int p_1=1$.

- Efficacy-рекурсия стартует с $p_1^{eff}(x)=\varphi(x;0,t_1)$ — это $B(t_1)\sim\mathcal N(0,t_1)$ под $H_0$.
- Futility-рекурсия стартует с $p_1^{fut}(x)=\varphi(x;\theta t_1,t_1)$ — это $B(t_1)\sim\mathcal N(\theta t_1,t_1)$ под $H_1$.


**Шаг 2. Отсечения (взгляд $k$)**

* Обрезаем Efficacy $\int_{b_k}^{\infty}p_k^{eff}(x)\,dx=\Delta\alpha_k$
* Обрезаем Futility $\int_{-\infty}^{a_k}p_k^{fut}(x)\,dx=\Delta\beta_k$

  <details>
  <summary>Подробнее</summary>

  * Efficacy — отрезаем правый хвост под $H_0$.

    Ошибка I рода = «объявили эффект, когда его нет». Под $H_0$ дрейф равен нулю, поэтому процесс $B$ «топчется» около нуля; ложноположительный исход — это траектория, которая забралась аномально высоко вверх (выше $b_k$). Вероятность такого события на взгляде $k$ — это масса правого хвоста, и мы приравниваем её к приращению бюджета:

  $$\int_{b_k}^{\infty}p_k^{eff}(x)\,dx=\Delta\alpha_k$$


  * Futility — отрезаем левый хвост под $H_1$.

    Ошибка II рода здесь = «зря остановили тест, когда эффект на самом деле есть». Под $H_1$ дрейф положителен ($\theta t_k$), поэтому «эффект есть» означает, что процесс в среднем дрейфует вверх. «Ошибочно выглядит слабо» — это траектория, которая оказалась аномально низко (ниже $a_k$). Вероятность такого события — масса левого хвоста:

  $$\int_{-\infty}^{a_k}p_k^{fut}(x)\,dx=\Delta\beta_k$$

  </details>


**Шаг 3. Свёртка**
  
  Оставляем только траектории в области продолжения и добавляем независимое приращение $B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta_{k+1},\Delta_{k+1})$, $\Delta_{k+1}=t_{k+1}-t_k$:

$$p_{k+1}^{fut}(y)=\int_{a_k}^{b_k}p_k^{fut}(x)\varphi\big(y,x+\theta\Delta_{k+1},\Delta_{k+1}\big)dx$$
$$p_{k+1}^{eff}(y)=\int_{a_k}^{b_k}p_k^{eff}(x)\varphi\big(y,x+\theta\Delta_{k+1},\Delta_{k+1}\big)dx$$

  <details>
  <summary>Подробнее</summary>

  Т.е. для каждой возможной текущей позиции $x$ берем гауссиану с центром $x+\theta\Delta$ и дисперсией $\Delta$, взвешенную тем, сколько траекторий находится в точке $x$; затем складываем все эти гауссианы и получаем новую плотность $p_{k+1}$.

  После отсечения на взгляде $k$ выжившие траектории распределены как $p_k(x)$, ограниченная областью продолжения $(a_k,b_k]$. Каждая выжившая траектория, находящаяся в точке $x$, на следующем шаге делает случайный независимый шаг:

  $$\text{следующая позиция} = x + \varepsilon,\qquad \varepsilon = B(t_{k+1})-B(t_k)\sim\mathcal N(\theta\Delta,\Delta).$$

  То есть мы буквально прибавляем $\varepsilon$ к позиции $x$ каждой траектории. Шаг $\varepsilon$ независим от прошлого — это то самое свойство из §1.3, ради которого вводили $B$. Раз он независим, его распределение зависит только от длины интервала $\Delta$, а не от всей истории.

  Отсюда и формула: траектория в точке $x$ с весом $p_k(x)dx$ порождает маленькую гауссиану $\varphi(y;x+\theta\Delta,\Delta)$, а новая плотность — интеграл по всем $x$:

  $$p_{k+1}(y)=\int_{a_k}^{b_k}p_k(x)\,\varphi\big(y;\,x+\theta\Delta,\Delta\big)\,dx.$$

  В результате форма распределения сдвигается вправо на $\theta\Delta$ (дрейф накопился) и размазан на $\Delta$ (шум накопился). Причём раз урезали $p_k$ до области продолжения, масса, вытекшая за границы, потеряна — поэтому $p_{k+1}$ интегрируется уже в меньшую величину (новую вероятность выживания). Каждый шаг рекурсии буквально «вычёркивает» остановившихся и «протаскивает» выживших на один взгляд вперёд.
  
  </details>

4. После построения границ их необходимо перевести обратно в Z шкалу:
$$c_k=\frac{b_k}{\sqrt{t_k}},\qquad a_k^{(\text{Z})}=\frac{a_k}{\sqrt{t_k}}.$$

Почему это работает. Урезание оставляет только «ещё продолжающиеся» траектории, а свёртка прибавляет следующий независимый шаг. Независимость приращений гарантирует, что распределение следующего шага зависит только от текущей позиции, а не от всей истории, — поэтому рекурсия точна. Каждый шаг «тратит» ровно приращение бюджета $\Delta\alpha_k$ или $\Delta\beta_k$; суммируясь по $k$, они дают в точности $\alpha$ и $\beta$.

In [25]:
def alpha_spending(t, alpha=0.05, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return 2 - 2 * norm.cdf(norm.ppf(1 - alpha / 2) / np.sqrt(t))
    if family == "pocock":
        return alpha * np.log(1 + (np.e - 1) * t)
    raise ValueError(family)


def beta_spending(t, beta=0.20, family="obf"):
    t = np.asarray(t, dtype=float)
    if family == "obf":
        return beta * t ** 3
    if family == "pocock":
        return beta * t
    raise ValueError(family)


def _convolve(p, grid, loc, scale):
    dx = grid[1] - grid[0]
    K = norm.pdf(grid[:, None], loc=grid[None, :] + loc, scale=scale)
    return (K @ p) * dx


def _quantile(p, grid, target, tail):
    dx = grid[1] - grid[0]
    cdf = np.cumsum(p) * dx
    vals = (cdf[-1] - cdf) if tail else cdf
    return grid[int(np.argmin(np.abs(vals - target)))]


def group_sequential_boundaries(t, alpha=0.05, beta=0.20, theta=None,
                                fam_alpha="obf", fam_beta="obf", grid_dx=0.01, n_sigma=6.0):
    t = np.asarray(t, dtype=float)
    K = len(t)
    if theta is None:
        theta = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    dt = np.diff(np.concatenate(([0.0], t)))
    grid = np.arange(-n_sigma, n_sigma + grid_dx, grid_dx)
    d_alpha = np.diff(np.concatenate(([0.0], alpha_spending(t, alpha, fam_alpha))))
    d_beta = np.diff(np.concatenate(([0.0], beta_spending(t, beta, fam_beta))))
    c = np.empty(K); a = np.empty(K)
    p_eff = norm.pdf(grid, loc=0.0, scale=math.sqrt(dt[0]))
    p_fut = norm.pdf(grid, loc=theta * dt[0], scale=math.sqrt(dt[0]))
    for k in range(K):
        b_k = _quantile(p_eff, grid, d_alpha[k], tail=True)
        c[k] = b_k / math.sqrt(t[k])
        a_k = _quantile(p_fut, grid, d_beta[k], tail=False)
        a[k] = a_k / math.sqrt(t[k])
        if k < K - 1:
            d = dt[k + 1]
            p_eff = _convolve(p_eff * (grid <= b_k), grid, 0.0, math.sqrt(d))
            p_fut = _convolve(p_fut * ((grid > a_k) & (grid <= b_k)), grid, theta * d, math.sqrt(d))
    return c, a


c_bound, a_bound = group_sequential_boundaries(design.t, design.alpha, design.beta, design.theta_mde)
pd.DataFrame({
    "взгляд k": np.arange(1, design.K + 1),
    "t_k": design.t,
    "c_k (efficacy)": np.round(c_bound, 3),
    "a_k (futility)": np.round(a_bound, 3),
})


ValueError: All arrays must be of the same length

In [ ]:
tt = np.asarray(design.t)
fig = go.Figure()
fig.add_trace(go.Scatter(x=tt, y=alpha_spending(tt, design.alpha, "obf"), mode="lines+markers",
                         line=dict(color=C_EFF, width=2.5), name="α-spending (OBF)"))
fig.add_trace(go.Scatter(x=tt, y=beta_spending(tt, design.beta, "obf"), mode="lines+markers",
                         line=dict(color=C_FUT, width=2.5), name="β-spending (ρ=3)"))
fig.add_hline(y=design.alpha, line_dash="dash", line_color=C_EFF, annotation_text="бюджет α")
fig.add_hline(y=design.beta, line_dash="dash", line_color=C_FUT, annotation_text="бюджет β")
fig = style(fig, title="Две стороны одной монеты: как «тратится» ошибка I и II рода",
            xlabel="доля информации t", ylabel="накопленная ошибка")
fig


In [ ]:
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
# зона futility (ниже a_k) и зона efficacy (выше c_k)
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[a_bound, bottom, bottom],
                         fill="toself", fillcolor="rgba(217,95,2,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[c_bound, top, top],
                         fill="toself", fillcolor="rgba(27,158,119,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy c_k"))
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="futility a_k"))
fig.add_annotation(x=0.66, y=3.4, text="СТОП — эффект", showarrow=False, font=dict(color=C_EFF, size=13))
fig.add_annotation(x=0.66, y=-2.4, text="СТОП — futility", showarrow=False, font=dict(color=C_FUT, size=13))
fig.add_annotation(x=0.66, y=0.9, text="продолжаем", showarrow=False, font=dict(size=13))
fig = style(fig, height=480, title="Двухграничный групповой-последовательный дизайн",
            xlabel="доля информации t", ylabel="Z(t)")
fig.update_yaxes(range=[bottom, top])
fig


Для двустороннего теста: efficacy — $|Z_t|>c_k$ (α делится на два хвоста), futility — $|Z_t|<a_k$ (эффект «зажат» в коридоре вокруг нуля). Геометрически futility — «воронка» вокруг нуля, efficacy — два симметричных хвоста.

In [ ]:
z_alpha2 = norm.ppf(1 - design.alpha / 2)
theta2 = z_alpha2 + norm.ppf(1 - design.beta)
c2, a2 = group_sequential_boundaries(design.t, design.alpha, design.beta, theta2)
tt = np.asarray(design.t)
bottom, top = -4.2, 4.2

fig = go.Figure()
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[a2, -a2[::-1], a2[0]],
                         fill="toself", fillcolor="rgba(217,95,2,0.14)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False, name="futility"))
fig.add_trace(go.Scatter(x=tt, y=c2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="± efficacy"))
fig.add_trace(go.Scatter(x=tt, y=-c2, mode="lines+markers", line=dict(color=C_EFF, width=2.8), showlegend=False))
fig.add_trace(go.Scatter(x=tt, y=a2, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="± futility"))
fig.add_trace(go.Scatter(x=tt, y=-a2, mode="lines+markers", line=dict(color=C_FUT, width=2.8), showlegend=False))
fig = style(fig, height=480, title="Двусторонний дизайн: futility = коридор |Z| < a_k",
            xlabel="доля информации t", ylabel="Z(t)")
fig.update_yaxes(range=[bottom, top])
fig


#### 4.2 Binding и non-binding фиксация.

<details>
<summary>Идея и доказательство (развернуть)</summary>

Futility-граница влияет на то, какие траектории доживают до конца. Вопрос: как учитывать её при расчёте α?

- **Binding**: futility-стоп обязателен, α считается с удалением траекторий ниже $a_k$. Если проигнорировать стоп — α вырастет.
- **Non-binding**: α считается так, *будто futility-границы нет*. Стоп можно отменить без риска для α.

**Почему binding α ≤ non-binding.** В non-binding область продолжения — «ещё не пересекли efficacy» ($-\infty<Z\le c_j$); в binding она дополнительно ограничена снизу ($a_j<Z\le c_j$). Продолжающееся множество в binding — подмножество, поэтому масса, достигающая efficacy, не больше.

</details>

График ниже показывает разницу: слева non-binding (зона продолжения — всё ниже efficacy-границы), справа binding (зона продолжения сужена снизу futility-границей). В обоих случаях считаем α efficacy-границы.


In [ ]:
def overall_alpha_upper(c, t, lower=None, n_sim=400000):
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(0, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    active = np.ones(n_sim, dtype=bool)
    crossed = np.zeros(n_sim, dtype=bool)
    for k in range(len(t)):
        if lower is not None:
            active &= ~(Z[:, k] < lower[k]) & ~crossed
        crossed |= active & (Z[:, k] > c[k])
        active &= ~crossed
    return crossed.mean()


t = np.asarray(design.t)
alpha_nonb = overall_alpha_upper(c_bound, t, lower=None)
alpha_bind = overall_alpha_upper(c_bound, t, lower=a_bound)

tt = np.asarray(design.t)
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Non-binding: α = {alpha_nonb:.4f}", f"Binding: α = {alpha_bind:.4f}"))

# non-binding: зона продолжения = всё ниже efficacy
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy"), row=1, col=1)
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.5, dash="dot"),
                         name="futility (игнорируется в α)"), row=1, col=1)

# binding: зона продолжения = между a_k и c_k
fig.add_trace(go.Scatter(x=np.r_[tt, tt[-1], tt[0]], y=np.r_[c_bound, a_bound[::-1], c_bound[0]],
                         fill="toself", fillcolor="rgba(153,153,153,0.15)", line=dict(width=0),
                         hoverinfo="skip", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=tt, y=c_bound, mode="lines+markers", line=dict(color=C_EFF, width=2.8), name="efficacy", showlegend=False), row=1, col=2)
fig.add_trace(go.Scatter(x=tt, y=a_bound, mode="lines+markers", line=dict(color=C_FUT, width=2.8), name="futility", showlegend=False), row=1, col=2)

fig.update_layout(height=440, title="Разница binding и non-binding в зоне продолжения (и в итоговом α)",
                  template="plotly_white")
fig.update_xaxes(title_text="t", showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig.update_yaxes(title_text="Z(t)", showgrid=True, gridcolor="rgba(0,0,0,0.07)")
fig


## 2. Влияние futility на ошибки I/II рода

<details>
<summary>Почему futility НЕ раздувает α, но снижает мощность (развернуть)</summary>

**Futility не раздувает α.** Ошибка I рода — траектория, которая под $H_0$ забралась выше efficacy. Futility отбрасывает траектории *ниже* себя; они и так редко достигали бы efficacy, а стоп лишь уменьшает их число. Вероятность отвергнуть при $H_0$ может только уменьшиться. Аргумент использует вложенность и монотонность, а не нормальность.

**Futility снижает мощность.** Под $H_1$ часть траекторий на раннем взгляде случайно выглядит слабо — futility зря их останавливает. Это плата за экономию трафика.

</details>

Симулируем групповой-последовательный дизайн и измеряем операционные характеристики. Ниже — результаты с человеческими названиями.


In [ ]:
def simulate_trial(theta, t, c, a, n_sim=50000, futility="boundary", gamma=0.15,
                    theta_cp=None, theta0=0.0, tau=0.8, z_alpha=1.6449):
    """Симуляция группового-последовательного дизайна.

    futility: "none" | "boundary" (фиксированная a_k) | "cp" (CP<gamma) | "ppos" (PPoS<gamma).
    Исходы: efficacy / futility / complete; возвращает исход и долю информации на остановке.
    """
    dt = np.diff(np.concatenate(([0.0], t)))
    B = np.cumsum(rng.normal(theta * dt, np.sqrt(dt), size=(n_sim, len(t))), axis=1)
    Z = B / np.sqrt(t)
    outcome = np.full(n_sim, "complete")
    stop_frac = np.ones(n_sim)
    for k in range(len(t)):
        active = outcome == "complete"
        if not active.any():
            break
        eff = active & (Z[:, k] > c[k])
        still = active & ~eff
        if k == len(t) - 1:
            # финальный взгляд: futility = Z < a_K (фиксированная граница);
            # CP/PPoS тут вырождены (t=1 → sqrt(1-t)=0), поэтому их не считаем.
            fut = still & (Z[:, k] < a[k])
        elif futility == "boundary":
            fut = still & (Z[:, k] < a[k])
        elif futility == "cp":
            cp = conditional_power(Z[still, k], t[k], theta_cp, z_alpha)
            fut = still.copy(); fut[still] = cp < gamma
        elif futility == "ppos":
            pp = ppos(Z[still, k], t[k], theta0, tau, z_alpha)
            fut = still.copy(); fut[still] = pp < gamma
        else:
            fut = np.zeros_like(still)
        outcome[eff] = "efficacy"; outcome[fut] = "futility"
        stop_frac[eff] = t[k]; stop_frac[fut] = t[k]
    return outcome, stop_frac


def oc(theta, t, c, a, n_sim=50000, **kw):
    out, stop = simulate_trial(theta, t, c, a, n_sim, **kw)
    return {
        "FPR / power": (out == "efficacy").mean(),
        "P(futility stop)": (out == "futility").mean(),
        "P(дошли до конца)": (out == "complete").mean(),
        "E[N]/N_max": stop.mean(),
    }


In [ ]:
t = np.asarray(design.t)
n_sim = 50000

rows = []
for label, theta in [("H0: эффекта нет (θ = 0)", 0.0), ("H1: эффект = MDE (θ = θ_MDE)", design.theta_mde)]:
    r = oc(theta, t, c_bound, a_bound, n_sim)
    r["Сценарий"] = label
    rows.append(r)

mc = pd.DataFrame(rows)[["Сценарий", "FPR / power", "P(futility stop)", "P(дошли до конца)", "E[N]/N_max"]]
mc = mc.rename(columns={"FPR / power": "Вероятность efficacy (под H0 — FPR, под H1 — мощность)"})
mc.round(4)


In [ ]:
# Операционные характеристики по спектру эффектов: три доли, в сумме = 1 (stacked area).
thetas = np.linspace(-0.2, 3.0, 17)
N = 30000
out_df = pd.DataFrame([oc(th, t, c_bound, a_bound, N) for th in thetas])
out_df["θ"] = thetas

fig = go.Figure()
fig.add_trace(go.Scatter(x=thetas, y=out_df["FPR / power"], mode="lines", stackgroup="one",
                         line=dict(color=C_EFF, width=0), fillcolor="rgba(27,158,119,0.6)", name="efficacy (power/FPR)"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(futility stop)"], mode="lines", stackgroup="one",
                         line=dict(color=C_FUT, width=0), fillcolor="rgba(217,95,2,0.6)", name="futility stop"))
fig.add_trace(go.Scatter(x=thetas, y=out_df["P(дошли до конца)"], mode="lines", stackgroup="one",
                         line=dict(color=C_CONT, width=0), fillcolor="rgba(153,153,153,0.5)", name="дошли до конца"))
fig.add_vline(x=design.theta_mde, line_dash="dash", line_color="black", annotation_text="θ_MDE")
fig = style(fig, height=460,
            title="Операционные характеристики: три полосы в сумме дают 1 (при любом θ исход ровно один)",
            xlabel="истинный дрейф θ", ylabel="доля исходов")
fig


In [ ]:
# Распределение исхода: столбцы под H0 и H1 (понятнее, чем сотни линий ансамбля).
n_sim = 50000
res0 = oc(0.0, t, c_bound, a_bound, n_sim)
res1 = oc(design.theta_mde, t, c_bound, a_bound, n_sim)

fig = go.Figure()
cats = ["efficacy", "futility", "дошли до конца"]
fig.add_trace(go.Bar(x=cats, y=[res0["FPR / power"], res0["P(futility stop)"], res0["P(дошли до конца)"]],
                     name="H0 (θ = 0)", marker_color=C_CONT))
fig.add_trace(go.Bar(x=cats, y=[res1["FPR / power"], res1["P(futility stop)"], res1["P(дошли до конца)"]],
                     name="H1 (θ = MDE)", marker_color=C_EFF))
fig = style(fig, height=420, title="Распределение исходов под H0 и H1",
            xlabel="исход", ylabel="вероятность")
fig.update_yaxes(range=[0, 1])
fig


## 8. Практические рекомендации

<details>
<summary>Развернуть</summary>

1. **Взгляды — не раньше 30–50% данных.** При малом $t$ карта CP почти плоская, ранний стоп зря теряет мощность.
2. **Порог CP/PPoS — 10–20%.** Выше — потеря мощности, ниже 5% — экономии почти нет.
3. **Выбор θ для CP** — как в разделе 2.
4. **Non-binding — дефолт:** стоп можно отменить, не испортив α.
5. **Полный дизайн = α-spending + β-spending.**

</details>


## 9. Сквозной пример: сравнение всех методов

Теперь соберём всё вместе на конкретном примере. Метрика — **средний чек** ($\sigma$ варьируемо, контроль $\mu_0$ не влияет на стандартизованный результат). Сравним, как разные правила остановки ведут себя по мощности, ошибке I рода и экономии трафика.

Два замечания к таблице: **(1)** binding и non-binding здесь используют одни и те же границы — они различаются только расчётом α (у binding он ниже), мощность и ASN у них совпадают; **(2)** PPoS-futility со **скептическим** prior ($\theta_0=0$, $\tau=0.8$) очень консервативен — видно, как сильно prior влияет на результат; с «дизайновым» prior ($\theta_0=\theta_{MDE}$) он близок к CP.


In [ ]:
def compare_methods(sigma=100.0, mde=10.0, alpha=0.05, beta=0.20, K=5,
                     fam_alpha="obf", fam_beta="obf", gamma=0.15, theta0=0.0, tau=0.8, n_sim=20000):
    delta = mde / sigma
    theta_mde = norm.ppf(1 - alpha) + norm.ppf(1 - beta)
    z_alpha = norm.ppf(1 - alpha)
    t = np.linspace(0.3, 1.0, K)
    c, a = group_sequential_boundaries(t, alpha, beta, theta_mde, fam_alpha, fam_beta)

    methods = {}
    # 1) фиксированный z-тест (один взгляд в конце)
    methods["Фиксированный z-тест"] = dict(t=np.array([1.0]), c=np.array([z_alpha]), a=np.array([-np.inf]), futility="none")
    # 2) GST только efficacy
    methods["GST (efficacy only)"] = dict(t=t, c=c, a=np.full(K, -np.inf), futility="none")
    # 3) GST + futility non-binding
    methods["GST + futility (non-binding)"] = dict(t=t, c=c, a=a, futility="boundary")
    # 4) GST + futility binding (те же границы, но α считаем binding)
    methods["GST + futility (binding)"] = dict(t=t, c=c, a=a, futility="boundary", binding=True)
    # 5) GST + CP-futility
    methods["GST + CP-futility"] = dict(t=t, c=c, a=a, futility="cp", gamma=gamma, theta_cp=theta_mde)
    # 6) GST + PPoS-futility
    methods["GST + PPoS-futility"] = dict(t=t, c=c, a=a, futility="ppos", gamma=gamma, theta0=theta0, tau=tau)

    rows = []
    for name, m in methods.items():
        r0 = oc(0.0, m["t"], m["c"], m["a"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        r1 = oc(theta_mde, m["t"], m["c"], m["a"], n_sim, futility=m["futility"],
                gamma=m.get("gamma", 0.15), theta_cp=m.get("theta_cp", theta_mde),
                theta0=m.get("theta0", 0.0), tau=m.get("tau", 0.8))
        fpr = r0["FPR / power"]
        if m.get("binding"):
            fpr = overall_alpha_upper(m["c"], m["t"], lower=m["a"], n_sim=200000)
        rows.append({
            "Метод": name,
            "FPR (H0)": fpr,
            "Мощность (H1)": r1["FPR / power"],
            "P(futility | H1)": r1["P(futility stop)"],
            "E[N]/N_max (H1)": r1["E[N]/N_max"],
        })
    df = pd.DataFrame(rows)
    return df, theta_mde


cmp_df, theta_mde = compare_methods()
cmp_df.round(4)


In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["Мощность (H1)"], name="Мощность (H1)", marker_color=C_EFF))
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["E[N]/N_max (H1)"], name="E[N]/N_max (экономия)", marker_color=C_CONT))
fig.add_trace(go.Bar(x=cmp_df["Метод"], y=cmp_df["FPR (H0)"], name="FPR (H0)", marker_color=C_FUT, visible="legendonly"))
fig = style(fig, height=480, title="Сравнение методов: мощность, экономия трафика и ошибка I рода",
            xlabel="метод", ylabel="значение")
fig.update_xaxes(tickangle=-20)
fig


In [ ]:
def plot_comparison(K=5, sigma=100.0, mde=10.0, fam_alpha="obf", fam_beta="obf", gamma=0.15):
    df, _ = compare_methods(sigma=sigma, mde=mde, K=K, fam_alpha=fam_alpha, fam_beta=fam_beta, gamma=gamma, n_sim=8000)
    fig = go.Figure()
    fig.add_trace(go.Bar(x=df["Метод"], y=df["Мощность (H1)"], name="Мощность (H1)", marker_color=C_EFF))
    fig.add_trace(go.Bar(x=df["Метод"], y=df["E[N]/N_max (H1)"], name="E[N]/N_max", marker_color=C_CONT))
    fig.add_trace(go.Bar(x=df["Метод"], y=df["FPR (H0)"], name="FPR (H0)", marker_color=C_FUT, visible="legendonly"))
    fig = style(fig, height=460, title=f"Сравнение методов (K={K}, σ={sigma}, MDE={mde})",
                xlabel="метод", ylabel="значение")
    fig.update_xaxes(tickangle=-20)
    return go.FigureWidget(fig)

interact(plot_comparison,
         K=IntSlider(value=5, min=1, max=50, step=1, description="взглядов K"),
         sigma=FloatSlider(value=100.0, min=10, max=500, step=10, description="σ метрики"),
         mde=FloatSlider(value=10.0, min=1, max=50, step=1, description="MDE"),
         fam_alpha=Dropdown(options=["obf", "pocock"], value="obf", description="α-spending"),
         fam_beta=Dropdown(options=["obf", "pocock"], value="obf", description="β-spending"),
         gamma=FloatSlider(value=0.15, min=0.05, max=0.40, step=0.05, description="порог CP"))


## 11. Итоги

1. **Futility не раздувает ошибку I рода** — в отличие от peeking; она лишь снижает мощность.
2. **CP** — шанс дойти до значимости при фиксированном θ; **PPoS** усредняет по неопределённости θ; CP — частный случай PPoS.
3. **Stochastic curtailment** — общая рамка CP и PPoS.
4. **β-spending — зеркало α-spending**: априорная futility-граница с контролем ошибки II рода.
5. **Non-binding** — безопасный дефолт.

| | CP | PPoS | β-spending |
|---|---|---|---|
| Философия | частотная | байесовская | частотная (дизайн) |
| Эффект | фиксированный θ | prior на θ | θ_MDE |
| Граница | данные-зависимая | данные-зависимая | фиксирована заранее |
| Контролирует | правило «стоп» | правило «стоп» | ошибку II рода |
| Когда | быстро оценить шансы | учесть неопределённость | спроектировать тест |
